# List the tables in the database to verify

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/code')

# ---------------------------------------
# 宏观因子 -- Download from Michael W. McCracken 
# ---------------------------------------

In [2]:
target_timeline = pd.date_range(start='1989-01-01', end='2024-12-31', freq='ME')
macro_config = pd.read_excel('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor.xlsx', sheet_name='McCracken', index_col=0)
macro_cols = macro_config.index.dropna().unique().tolist()
# 构建承载最终结果的大面板 (初始化全为 NaN)
master_panel = pd.DataFrame(index=target_timeline, columns=macro_cols)
master_panel.index.name = 'mthcaldt'

print(f">>> Target Timeline: 1989-01 to 2024-12 ({len(target_timeline)} months)")
vintages_dir='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/Historical FRED-MD Vintages Final'
output_save_path='/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data'

print(f">>> 开始动态构建宏观大表 (共 {len(target_timeline)} 个月)...")

for cutoff_date in target_timeline:
    # A. 1999年8月前借用 1999-08.csv，之后直接对齐 YYYY-MM.csv
    if cutoff_date < pd.to_datetime('1999-08-31'):
        continue
    else:
        filename = cutoff_date.strftime('%Y-%m') + ".csv"
        
    file_path = os.path.join(vintages_dir, filename)

    # B. 读入单张快照大表
    df_raw_v = pd.read_csv(file_path)
    transform_dict = df_raw_v.iloc[0].to_dict() # 转换指令
    df_vintage = df_raw_v.iloc[1:].copy()        # 真实数据
    
    df_vintage['sasdate'] = pd.to_datetime(df_vintage['sasdate']) + pd.offsets.MonthEnd(0)  # 确保日期是月末
    df_vintage.set_index('sasdate', inplace=True)
    
    # 强制转成浮点数
    df_vintage = df_vintage.apply(pd.to_numeric, errors='coerce')
    
    # C. 自动化平稳化变形 (根据指令自动洗干净整张表的历史)
    for col in macro_cols:
        if col in df_vintage.columns:
            try: t_code = int(float(transform_dict[col]))
            except: t_code = 1
            
            if t_code == 1 or col in ['GS10', 'TB3MS', 'BAA']: pass
            elif t_code == 2: df_vintage[col] = df_vintage[col].diff()
            elif t_code == 3: df_vintage[col] = df_vintage[col].diff().diff()
            elif t_code == 4: df_vintage[col] = np.log(df_vintage[col])
            elif t_code == 5: df_vintage[col] = np.log(df_vintage[col]).diff()
            elif t_code == 6: df_vintage[col] = np.log(df_vintage[col]).diff().diff()
            else: df_vintage[col] = df_vintage[col].pct_change(fill_method=None)

            current_view = master_panel.loc[:cutoff_date, col]
            n_missing = current_view.isna().sum() 

            valid_vintage_data = df_vintage[col].dropna()
            n_take = min(n_missing, len(valid_vintage_data))
            # copy the last n_missing valid data points from the vintage to fill the missing values in the master panel without using index
            master_panel.loc[current_view[current_view.isna()].index, col] = valid_vintage_data.tail(n_take).values

# ==========================================================
# 🛠️ 衍生计算：在终极无偏大表生成后，横向合成两个利差
# ==========================================================
print(">>> 正在横向合成期限利差与信用利差...")
MacroFactor_McCracken = master_panel.reset_index()

MacroFactor_McCracken['TERM_SPREAD'] = MacroFactor_McCracken['GS10'] - MacroFactor_McCracken['TB3MS']
MacroFactor_McCracken['CREDIT_SPREAD'] = MacroFactor_McCracken['BAA'] - MacroFactor_McCracken['GS10']

# 计算完利差后，物理扔掉绝对值指标，保持特征纯净
MacroFactor_McCracken.drop(columns=['GS10', 'TB3MS', 'BAA'], inplace=True, errors='ignore')

# 导出为parquet
MacroFactor_McCracken.to_parquet(os.path.join(output_save_path, 'MacroFactor_McCracken.parquet'), index=False)

print(f"Total missing values: {MacroFactor_McCracken.isna().sum().sum()}")
display(MacroFactor_McCracken.head(5))

>>> Target Timeline: 1989-01 to 2024-12 (432 months)
>>> 开始动态构建宏观大表 (共 432 个月)...
>>> 正在横向合成期限利差与信用利差...
Total missing values: 0


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1989-01-31,0.003032,0.0,0.023065,0.00509,0.002328,0.093209,0.0,7.354362,-0.002089,0.040504,0.008734,0.41,0.45,0.3,-0.000011,17.7542,1.89,1.04,1.54
1,1989-02-28,0.002827,0.1,-0.050094,0.005545,0.005512,-0.029077,-0.01,7.390799,-0.001539,-0.003615,-0.006607,0.36,-0.07,-0.3,0.000814,17.758,1.53,0.82,1.56
2,1989-03-31,0.002343,-0.2,0.050094,-0.007935,0.001032,-0.030403,0.0,7.261927,-0.000801,-0.042,0.018435,0.24,0.33,-0.2,-0.000838,18.3126,1.25,0.64,1.44
3,1989-04-30,0.001807,-0.2,0.035204,0.009347,-0.002196,-0.001593,0.03,7.25982,0.00223,-0.014152,-0.022337,0.49,0.58,-0.1,0.001626,17.4754,0.82,0.54,1.31
4,1989-05-31,0.001851,0.2,-0.054941,0.002448,-0.000903,0.008623,0.01,7.199678,-0.000504,0.011182,0.006567,-0.01,-0.14,0.2,0.002414,16.8875,0.77,0.53,1.43


# --------------------------------------------
# 微观因子 -- WRDS CRSP Ratios 
# --------------------------------------------

这是量化基本面研究中最正统、也最纯粹的武器库——**WRDS Financial Ratios Suite (WRDS 官方财务比率库)**。

对于你目前暂时搁置 OAP 异象数据、决定专心用基本面和宏观数据跑通 MDN 基线模型（Baseline）的战略来说，这张表简直完美。它最大的价值在于：WRDS 官方已经帮你把极其复杂的底层财报科目，计算成了**对冲基金最常用的标准化比率**，并且完美对齐了 `public_date`（无前视偏差）。

面对这近 90 个变量，你绝对不能全塞进神经网络（会面临严重的多重共线性和过拟合）。结合你预测 **$ES_{5\%}$ (极左尾崩盘风险)** 的目标，我为你将这些指标分为 **5 大核心战区**，并为你挑出 MDN 最需要的“尾部特征”。

---

### 🚨 战区一：债务与破产雷区 (Solvency & Leverage) - **最核心的左尾触发器**
对于预测极值崩盘，**“公司欠了多少钱，还不还得上”**永远是第一位的。高杠杆公司在宏观环境恶化时，最容易出现断崖式下跌。

* **`short_debt` (短期债务 / 总债务):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 极度致命。短债比例越高，资金链断裂的风险呈指数级上升。
* **`intcov_ratio` (利息保障倍数):** **【MDN 必选 ⭐⭐⭐⭐】** EBIT / 利息支出。衡量公司赚的钱够不够还利息。低于 1 的公司随时暴雷。
* **`debt_at` (总债务 / 总资产):** **【MDN 必选 ⭐⭐⭐】** 最经典的宏观杠杆率。
* **`debt_ebitda` (总债务 / EBITDA):** 华尔街信用评级最看重的指标，比 `debt_at` 更能反映真实的偿债压力。
* *其他同类冗余指标 (建议剔除其一):* `lt_debt`, `debt_capital`, `de_ratio`, `totdebt_invcap`（选 `debt_at` 即可，没必要全放）。

### 🩸 战区二：流动性与现金流 (Liquidity & Cash Flow) - **抗风险的护城河**
当市场恐慌时，利润只是会计数字，**现金才是命**。缺乏流动性的公司在遭遇做空或行业寒冬时毫无抵抗力。

* **`ocf_lct` (经营现金流 / 流动负债):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 这是最真实的短期生存指标。赚不到真金白银还要面临短期债务的公司，左尾极厚。
* **`cash_ratio` (现金比率):** **【MDN 必选 ⭐⭐⭐⭐】** 账面纯现金 / 流动负债。直接衡量“今天如果债主逼债，能不能活下来”。
* **`fcf_ocf` (自由现金流 / 经营现金流):** 反映公司的资本开支压力。
* *其他同类冗余指标:* `curr_ratio` (流动比率), `quick_ratio` (速动比率)。在预测崩盘时，它们不如 `cash_ratio` 狠，因为存货和应收账款在破产时往往一文不值。

### 🎭 战区三：盈余质量 (Accounting Quality) - **财务造假与爆雷预测**
* **`accrual` (应计项目 / 平均资产):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 盈余操纵的试金石！如果一家公司净利润很高，但全在应收账款和存货里（高 accrual），这是未来业绩大爆雷（股价单日腰斩）的最强预兆。

### 💰 战区四：估值与均值回归 (Valuation & Multiples) - **决定下跌的空间**
估值越高，泡沫破裂时的“重力加速度”越大。估值因子是决定收益率分布中心 $\mu$ 和方差 $\sigma$ 的重要锚点。

* **`bm` (账面市值比):** **【MDN 必选 ⭐⭐⭐⭐⭐】** 最经典的价值因子。
* **`pe_exi` (扣非市盈率):** **【MDN 必选 ⭐⭐⭐⭐】** 剔除了非经常性损益的真实市盈率，比 `pe_inc` 更干净。
* **`evm` (企业价值倍数 EV/EBITDA):** 并购和私募股权最爱看的指标，不受资本结构（负债率）影响。
* *其他同类冗余指标:* `pe_op_basic`, `ptb`, `ps` (市销率), `pcf` (市现率)。选 2-3 个核心的即可。

### ⚙️ 战区五：盈利能力与效率 (Profitability & Efficiency)
* **`roe` (净资产收益率):** **【MDN 必选 ⭐⭐⭐⭐】** 基本面质量的终极体现。
* **`gpm` (毛利率) 或 `opmad` (税后营业利润率):** 衡量公司在行业内的护城河深度。毛利被压缩通常是崩盘的早期信号。
* **`at_turn` (总资产周转率):** 运营效率。
* *其他同类冗余指标:* `roa`, `roce`, `aftret_eq` 等高度相关，留 `roe` 足矣。

---

### 🗑️ 垃圾桶：绝对不要放入神经网络的变量

1.  **标识符与日期：** `gvkey`, `adate`, `qdate`, `public_date`（仅用于合并和排序，绝对不能作为 $X$ 输入网络）。
2.  **文本与描述：** `gicdesc`, `ffi48_desc` 等一切 `VARCHAR` 文本列。
3.  **绝对数值：** `mktcap` (总市值), `price` (股价)。**如果你要放，必须对它们进行横截面 Rank 转换**，否则几万亿的市值会瞬间引起梯度爆炸。

---

### 🚀 你的 MDN 基线特征包 (Baseline Feature Set) 建议

为了避免维度灾难，我建议你从这张表里**精挑细选这 15 个黄金比率**作为 $X_{Fundamentals}$：

```python
mdn_features_wrds = [
    # 破产与债务
    'short_debt', 'intcov_ratio', 'debt_at', 'debt_ebitda',
    # 流动性与造假预警
    'ocf_lct', 'cash_ratio', 'accrual', 
    # 估值泡沫
    'bm', 'pe_exi', 'evm', 'divyield',
    # 盈利与效率
    'roe', 'gpm', 'at_turn', 'rd_sale'  # rd_sale (研发占比) 对美股科技股防雷很有用
]
```

将这 15 个特征，加上你处理好的 Goyal 宏观特征（如 `TAIL`, `DEF`, `VIXCLS`），你的第一版输入层 $X_T$ 就能控制在 20 个维度左右。配合上横截面 Rank 标准化，这是一个极其清爽且具备极强经济学解释力的输入矩阵！

这些财务比率中必定存在大量的 `NaN`（比如很多科技公司没有短期债务，或者尚未盈利导致市盈率缺失）。面对这些残缺，在进行 Rank 转换之前，你打算使用全市场的截面中位数填充，还是基于它所在的行业 (`gsector` / `ffi48`) 进行同行业中位数填充？

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行
wrds_path = '/Users/jianbinchen/NonSync/GitHub/Research/DataSet/WRDS.sqlite'
# set current working directory to the location of the database file
os.chdir('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/code')

In [4]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow')
crsp_wrds_msfv2_query_1990

,permno,mthcaldt,mthret,mthretx,mthprc,mthcap,mthvol,shrout,naics,siccd,mthdelflg,delreasontype,year
0,10001,1990-01-31,-0.018519,-0.018519,9.9375,1.015613e+04,3.525500e+04,1022,0,4920,N,UNAV,1990
1,72996,1990-01-31,-0.106383,-0.106383,5.2500,1.897875e+04,7.820000e+04,3615,0,6211,N,NACT,1990
2,72988,1990-01-31,-0.076923,-0.076923,1.5000,2.658000e+03,1.315200e+04,1772,0,2099,N,LP,1990
3,72980,1990-01-31,-0.117647,-0.117647,15.0000,5.571000e+04,7.385600e+04,3714,0,6360,N,NACT,1990
4,11278,1990-01-31,-0.145161,-0.145161,6.6250,5.448400e+04,3.259640e+05,8224,0,5390,N,BKPY,1990
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2198469,93345,2025-12-31,-0.057804,-0.057804,1.6300,1.472281e+05,2.542434e+07,90324,325180,2819,N,NACT,2025
2198470,21122,2025-12-31,0.012152,0.012152,14.1600,1.702797e+06,2.911402e+07,120254,522320,6099,N,NACT,2025
2198471,21118,2025-12-31,-0.040892,-0.040892,2.5800,2.884208e+05,4.764334e+07,111791,424350,5137,N,NACT,2025
2198472,84397,2025-12-31,0.036761,0.033780,51.7200,5.192171e+05,9.384600e+05,10039,522110,6022,N,NACT,2025


In [5]:
crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow',columns=['permno', 'mthcaldt', 'mthret', 'mthprc', 'mthcap', 'mthvol', 'shrout', 'siccd'])
# crsp_wrds_msfv2_query_1990=pd.read_parquet('../data/crsp_wrds_msfv2_query_1990.parquet', engine='pyarrow')

# ----------------------------------------
# 剔除金融行业 (SIC 代码 6000 至 6999) 暂时作废
# crsp_wrds_msfv2_query_1990['siccd']=crsp_wrds_msfv2_query_1990['siccd'].astype('Int64')
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990[(crsp_wrds_msfv2_query_1990['siccd'] < 6000) | (crsp_wrds_msfv2_query_1990['siccd'] > 6999)]
# crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop(columns=['siccd'])
# ----------------------------------------

print(f"total rows: {len(crsp_wrds_msfv2_query_1990)}")
crsp_wrds_msfv2_query_1990=crsp_wrds_msfv2_query_1990.drop_duplicates(subset=['permno', 'mthcaldt'], keep='first')
print(f"total rows after drop duplicates: {len(crsp_wrds_msfv2_query_1990)}")
crsp_wrds_msfv2_query_1990.head(5)

total rows: 2198474
total rows after drop duplicates: 2192931


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022,4920
1,72996,1990-01-31,-0.106383,5.2500,18978.75,78200.0,3615,6211
2,72988,1990-01-31,-0.076923,1.5000,2658.00,13152.0,1772,2099
3,72980,1990-01-31,-0.117647,15.0000,55710.00,73856.0,3714,6360
4,11278,1990-01-31,-0.145161,6.6250,54484.00,325964.0,8224,5390


In [6]:
MicroFactor = pd.read_excel('../data/MicroFactor.xlsx', sheet_name='select')
MicroFactor.head(5)

,name,nullable,type,comment,description,select,中文解释
0,0,gvkey,True,VARCHAR(7),Global Company Key,1.0,NaN
1,1,adate,True,DATE,Fiscal year end,NaN,NaN
2,2,qdate,True,DATE,Fiscal quarter end,NaN,NaN
3,3,public_date,True,DATE,Date,1.0,"正常日历月, 比如1月份的时候, 能知道的各项数据"
4,4,capei,True,DOUBLE PRECISION,Shillers Cyclically Adjusted P/E Ratio,NaN,#Shiller CAPE，长期均值回归


In [7]:
selected_features = MicroFactor[MicroFactor['select']==1]['nullable'].tolist()
comp_wrds_ratios = pd.read_parquet('../data/comp_wrds_ratios.parquet', engine='pyarrow', 
                                   columns=selected_features)
comp_wrds_ratios.head(5)

,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49
0,001003,1990-01-31,NaN,-2.836651,-0.052521,0.034461,-0.159702,-0.139973,-0.276872,-0.153649,0.314643,-0.217813,-1.813523,-1.166551,0.488989,0.413638,-1.832834,0.984159,-0.128283,NaN,17.424393,-4.555412,0.056425,0.960262,1.565528,1.554107,11.552599,11.534909,28.633479,0.0,0.064253,0.0,-0.355052,NaN,43.0
1,013343,1990-01-31,3.597226,6.485974,4.511111,1.409560,2.241353,2.241353,0.299322,0.299322,0.299322,0.630680,0.041494,0.000000,0.040907,0.009276,0.085077,1.000000,NaN,1.000000,0.024770,315.769231,NaN,NaN,NaN,0.136665,47.824790,12.759404,0.140050,0.0,0.003948,0.0,-0.044903,NaN,48.0
2,002484,1990-01-31,0.715879,NaN,11.229942,0.360000,-17.721272,NaN,0.032242,0.047622,NaN,NaN,0.116036,NaN,NaN,0.135434,NaN,0.689936,-0.090208,NaN,0.900549,NaN,0.201836,1.769802,NaN,1.870810,99.869707,NaN,3.555566,0.0,NaN,NaN,0.099620,NaN,43.0
3,013342,1990-01-31,1.403200,NaN,7.986110,0.532836,53.053115,-138.661550,0.068557,0.079114,0.079114,0.092006,0.097703,0.007018,0.031130,0.041482,NaN,0.275118,NaN,-1.026087,2.493436,NaN,NaN,NaN,NaN,0.393489,2.560843,6.071107,1.406800,0.0,0.000000,0.0,0.023184,0.046377,46.0
4,013341,1990-01-31,0.468837,7.854662,9.400269,0.629987,1.736912,1.824186,0.074888,0.116919,0.128298,0.087049,0.247401,0.156030,0.071427,0.257980,3.611789,0.023124,NaN,0.952157,4.686480,3.942446,NaN,NaN,NaN,0.556728,5.808530,6.474053,3.165822,0.0,0.000000,0.0,-0.165359,0.012616,46.0


In [8]:
comp_wrds_ratios.duplicated(subset=['gvkey', 'public_date'], keep=False).sum()

np.int64(0)

In [9]:
# merge crsp_wrds_msfv2_query_1990 and comp_wrds_ratios by ccm link
with sqlite3.connect(wrds_path) as db_conn:
    crsp_ccmxpf_linktable = pd.read_sql('SELECT * FROM crsp_ccmxpf_linktable', db_conn, 
                                        parse_dates=['linkdt','linkenddt'])
crsp_ccmxpf_linktable['linkenddt'] = pd.to_datetime(crsp_ccmxpf_linktable['linkenddt']).fillna(pd.to_datetime('2099-12-31'))
crsp_ccmxpf_linktable['lpermno'] = crsp_ccmxpf_linktable['lpermno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
crsp_ccmxpf_linktable['gvkey'] = crsp_ccmxpf_linktable['gvkey'].astype(str).str.zfill(6)

crsp_wrds_msfv2_query_1990['permno'] = crsp_wrds_msfv2_query_1990['permno'].astype("Int64").astype(str).replace("<NA>", pd.NA).str.strip()
comp_wrds_ratios['gvkey'] = comp_wrds_ratios['gvkey'].astype(str).str.zfill(6)

ccm = crsp_ccmxpf_linktable[(crsp_ccmxpf_linktable['linktype'].isin(['LU', 'LC'])) & (crsp_ccmxpf_linktable['linkprim'].isin(['P', 'C']))]
ccm = ccm.rename(columns={'lpermno': 'permno'})

comp_ccm = pd.merge(comp_wrds_ratios, ccm[['gvkey', 'permno', 'linkdt', 'linkenddt']], on='gvkey', how='inner')
comp_ccm = comp_ccm[(comp_ccm['public_date'] >= comp_ccm['linkdt']) & 
                    (comp_ccm['public_date'] <= comp_ccm['linkenddt'])]

comp_ccm['public_date'] = comp_ccm['public_date'] + pd.offsets.MonthEnd(0) # 把日期推到月底，和CRSP日期对齐
crsp_wrds_msfv2_query_1990['mthcaldt'] = crsp_wrds_msfv2_query_1990['mthcaldt'] + pd.offsets.MonthEnd(0)# 把日期推到月底
print(f"CRSP monthly data shape: {crsp_wrds_msfv2_query_1990.shape}")
# 匹配每一行有交易数据的股票和财报数据，注意这里是 left join，保留所有的 CRSP 行，财报数据不匹配的会被填 NaN
micro_df = pd.merge(
    crsp_wrds_msfv2_query_1990,
    comp_ccm,
    left_on=['permno', 'mthcaldt'],
    right_on=['permno', 'public_date'],
    how='left'
)
micro_df = micro_df.sort_values(['permno', 'mthcaldt']).reset_index(drop=True)

# =========================================================
# 核心修复：强制对齐“日历月网格” (Resampling to Continuous Grid)
# =========================================================
print(" -> Reindexing panel data to a continuous monthly grid...")

# 2. 将日期设置为 Index，为 Pandas 的重采样做准备
micro_df = micro_df.set_index('mthcaldt')

# 3. ★ 核心魔法：按 permno 分组，按月 ('ME' 或 'M') 重新采样
# 只要两个日期之间有断层，Pandas 就会自动插入缺失的月份，并将那一行的值设为 NaN
micro_df = micro_df.groupby('permno').resample('ME').asfreq().drop(columns=['permno']).reset_index()
micro_df.head(5)

CRSP monthly data shape: (2192931, 8)
 -> Reindexing panel data to a continuous monthly grid...


,permno,mthcaldt,mthret,mthprc,mthcap,mthvol,shrout,siccd,gvkey,public_date,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,fcf_ocf,de_ratio,intcov_ratio,cash_ratio,curr_ratio,inv_turn,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,divyield,ffi49,linkdt,linkenddt
0,10001,1990-01-31,-0.018519,9.9375,10156.13,35255.0,1022.0,4920.0,012994,1990-01-31,0.917331,4.853678,2.743056,0.421441,7.769246,7.599586,0.050236,0.103437,0.134464,0.081263,0.151031,0.139773,0.164538,0.426239,2.590528,0.107954,0.339098,0.749436,2.235379,3.131479,0.271814,1.173138,91.308370,1.223659,7.855339,10.165835,3.959000,0.0,0.0,0.0,-0.005309,0.050633,31.0,1986-01-09,2017-08-31
1,10001,1990-02-28,-0.006289,9.8750,10092.25,14862.0,1022.0,4920.0,012994,1990-02-28,0.849363,4.626443,2.544529,0.403697,5.646409,7.695783,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31
2,10001,1990-03-31,0.012821,9.8750,10141.63,12727.0,1027.0,4920.0,012994,1990-03-31,0.849363,4.626443,2.576336,0.410743,5.744959,7.830102,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.049383,31.0,1986-01-09,2017-08-31
3,10001,1990-04-30,0.000000,9.8750,10141.63,16645.0,1027.0,4920.0,012994,1990-04-30,0.849363,4.626443,2.544529,0.405672,5.674033,7.733434,0.052654,0.107955,0.139398,0.084097,0.162685,0.141288,0.181426,0.413490,2.279116,0.088959,0.508677,0.749436,2.084969,3.119863,0.273871,1.243238,96.509413,1.301493,8.324893,10.832774,4.015067,0.0,0.0,0.0,-0.024148,0.050000,31.0,1986-01-09,2017-08-31
4,10001,1990-05-31,-0.012658,9.7500,10013.25,27885.0,1027.0,4920.0,012994,1990-05-31,0.911008,4.819588,2.480916,0.422964,4.424768,7.540098,0.056644,0.116415,0.148982,0.089212,0.158487,0.139023,0.182316,0.403169,2.211369,0.066927,0.693056,0.749436,1.943177,3.227166,0.263992,1.278309,89.841695,1.223747,8.168377,10.663666,3.601704,0.0,0.0,0.0,-0.047016,0.051282,31.0,1986-01-09,2017-08-31


In [10]:
# 按月统计全部股票每一列缺失的占比, 只打印超过20%的月份和列
missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())
missing_by_month = missing_by_month[missing_by_month > 0.2].stack().reset_index() #stack() 将多层索引的 DataFrame 转换为长格式
missing_by_month.columns = ['mthcaldt', 'column', 'missing_ratio']
print("Months and columns with more than 20% missing values:")
print(missing_by_month)
missing_by_month.to_excel('../data/missing_by_month.xlsx', index=False)
# 主要是divyield缺失比较严重

Months and columns with more than 20% missing values:
       mthcaldt       column  missing_ratio
0    1990-01-31       pe_exi       0.218412
1    1990-01-31           ps       0.202872
2    1990-01-31         pfcf       0.257601
3    1990-01-31          npm       0.202872
4    1990-01-31        opmad       0.203209
...         ...          ...            ...
3457 2025-12-31     pay_turn       0.205175
3458 2025-12-31  sale_equity       0.262676
3459 2025-12-31     adv_sale       0.217198
3460 2025-12-31   staff_sale       0.217198
3461 2025-12-31     divyield       0.635912

[3462 rows x 3 columns]


/var/folders/8b/0krd96r57jldyg4t2s79cv140000gn/T/ipykernel_59787/790039815.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_by_month = micro_df.groupby('mthcaldt').apply(lambda x: x.isna().mean())


In [11]:
# micro_df.drop(columns=[ 'public_date', 'linkdt', 'linkenddt', 'log_ret'], errors='ignore', inplace=True)
micro_df.to_parquet('../data/micro_data.parquet', index=False)